<a href="https://colab.research.google.com/github/fmaignacio/datajud_probe/blob/claude%2Fevaluate-legal-databases-yWU5P/notebooks/00b_download_stj_textos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00b - Download STJ Textos Integrais

Objetivo: baixar os arquivos ZIP de textos integrais da base STJ - Íntegras de Decisões Terminativas e Acórdãos do Diário da Justiça.

Este notebook complementa o `00_download_stj_metadados.ipynb`: os metadados ficam em `raw/stj_integras_metadata/` e os ZIPs de texto ficam em `raw/stj_integras/`, ambos organizados por ano.

## 1. Ambiente

In [1]:
# Rode no Colab se necessario.
# !pip install requests beautifulsoup4 tqdm pandas

In [2]:
from pathlib import Path
from urllib.parse import urljoin, unquote
import random
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [3]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    MOUNT_POINT = Path('/content/drive')
    if (MOUNT_POINT / 'MyDrive').exists():
        print('Google Drive ja esta montado.')
    else:
        drive.mount(str(MOUNT_POINT), force_remount=True)
    DATA_ROOT = MOUNT_POINT / 'MyDrive/Mestrado/2026/llms/data'
else:
    DATA_ROOT = Path.cwd() / 'data'

TEXTOS_RAW = DATA_ROOT / 'raw' / 'stj_integras'
REPORTS_DATA = DATA_ROOT / 'reports' / 'summaries'

for path in [TEXTOS_RAW, REPORTS_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT:', DATA_ROOT)
print('TEXTOS_RAW:', TEXTOS_RAW)

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/Mestrado/2026/llms/data
TEXTOS_RAW: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras


## 2. Descobrir links de ZIP

In [4]:
DATASET_URL = 'https://dadosabertos.web.stj.jus.br/dataset/integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 datajud-probe academic text downloader',
}

response = requests.get(DATASET_URL, headers=HEADERS, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

zip_urls = []
for a in soup.select('a[href]'):
    href = urljoin(DATASET_URL, a['href'])
    filename = unquote(href.split('/')[-1].split('?')[0])

    # O portal alterna entre URLs como 20240112.zip e 20240205,
    # exibindo o formato ZIP separado do nome. Aceitamos os dois.
    if re.fullmatch(r'(?:textos)?\d{6,8}(?:\.zip)?', filename):
        zip_urls.append(href)

zip_urls = sorted(set(zip_urls))

print('Arquivos ZIP encontrados:', len(zip_urls))
print('Primeiros:', [unquote(url.split('/')[-1]) for url in zip_urls[:5]])
print('Ultimos:', [unquote(url.split('/')[-1]) for url in zip_urls[-5:]])

Arquivos ZIP encontrados: 1196
Primeiros: ['textos20211011.zip', 'textos20240705.zip', '20231003.zip', 'textos20250806.zip', '20230217.zip']
Ultimos: ['20220627.zip', 'textos20240618.zip', 'textos20240619.zip', '20221227.zip', 'textos20210910.zip']


## 3. Selecionar recorte

In [5]:
# Corpus atual do projeto.
ANOS_ANALISE = [2021, 2022, 2023]

# Use None para nao limitar um lado do intervalo.
DATA_INICIO = '2021-01-01'
DATA_FIM = '2023-12-31'

# Para um teste pequeno, defina um numero. Para baixar todo o recorte selecionado, deixe None.
MAX_DOWNLOADS = None

# Mantem compatibilidade com os notebooks 02/07, que ja esperam nomes como textos20260319.zip.
RENOMEAR_COM_PREFIXO_TEXTOS = True
OVERWRITE = False


def zip_filename_from_url(url: str) -> str:
    return unquote(url.split('/')[-1].split('?')[0])


def date_from_zip_filename(filename: str) -> pd.Timestamp | None:
    match = re.fullmatch(r'(?:textos)?(\d{6}|\d{8})(?:\.zip)?', filename)
    if not match:
        return None
    raw_date = match.group(1)
    fmt = '%Y%m%d' if len(raw_date) == 8 else '%Y%m'
    return pd.to_datetime(raw_date, format=fmt, errors='coerce')


def output_zip_filename(filename: str) -> str:
    if not RENOMEAR_COM_PREFIXO_TEXTOS:
        return filename
    match = re.fullmatch(r'(?:textos)?(\d{6}|\d{8})(?:\.zip)?', filename)
    return f'textos{match.group(1)}.zip' if match else filename

start = pd.to_datetime(DATA_INICIO) if DATA_INICIO else None
end = pd.to_datetime(DATA_FIM) if DATA_FIM else None

selected_urls = []
for url in zip_urls:
    filename = zip_filename_from_url(url)
    url_date = date_from_zip_filename(filename)
    if url_date is None or pd.isna(url_date):
        continue
    if ANOS_ANALISE is not None and int(url_date.year) not in ANOS_ANALISE:
        continue
    if start is not None and url_date < start:
        continue
    if end is not None and url_date > end:
        continue
    selected_urls.append(url)

selected_urls = sorted(selected_urls, key=lambda url: date_from_zip_filename(zip_filename_from_url(url)))

if MAX_DOWNLOADS is not None:
    selected_urls = selected_urls[:MAX_DOWNLOADS]

print('Selecionados:', len(selected_urls))
print([zip_filename_from_url(url) for url in selected_urls[:10]])

Selecionados: 643
['textos20210104.zip', 'textos20210105.zip', 'textos20210106.zip', 'textos20210107.zip', 'textos20210111.zip', 'textos20210112.zip', 'textos20210113.zip', 'textos20210114.zip', 'textos20210115.zip', 'textos20210118.zip']


## 4. Download robusto

In [6]:
def download_zip_file(
    url: str,
    output_dir: Path,
    overwrite: bool = False,
    attempts: int = 4,
    base_sleep: float = 5.0,
) -> dict:
    source_filename = zip_filename_from_url(url)
    filename = output_zip_filename(source_filename)
    file_date = date_from_zip_filename(source_filename)
    year_dir = output_dir / str(file_date.year) if file_date is not None and not pd.isna(file_date) else output_dir / 'sem_data'
    year_dir.mkdir(parents=True, exist_ok=True)
    output_path = year_dir / filename

    if output_path.exists() and not overwrite:
        return {
            'url': url,
            'source_filename': source_filename,
            'filename': filename,
            'status': 'already_exists',
            'path': str(output_path),
            'bytes': output_path.stat().st_size,
            'error': None,
        }

    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            with requests.get(url, headers=HEADERS, stream=True, timeout=300) as response:
                response.raise_for_status()
                tmp_path = output_path.with_suffix(output_path.suffix + '.part')
                bytes_written = 0
                with tmp_path.open('wb') as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            bytes_written += len(chunk)
                tmp_path.replace(output_path)

            return {
                'url': url,
                'source_filename': source_filename,
                'filename': filename,
                'status': 'downloaded',
                'path': str(output_path),
                'bytes': bytes_written,
                'error': None,
            }
        except requests.HTTPError as exc:
            status_code = exc.response.status_code if exc.response is not None else None
            last_error = f'HTTP {status_code}: {exc}'
        except requests.RequestException as exc:
            last_error = repr(exc)

        sleep_seconds = base_sleep * attempt + random.uniform(0, 3)
        print(f'Falha em {source_filename} tentativa {attempt}/{attempts}: {last_error}. Aguardando {sleep_seconds:.1f}s')
        time.sleep(sleep_seconds)

    return {
        'url': url,
        'source_filename': source_filename,
        'filename': filename,
        'status': 'failed',
        'path': str(output_path),
        'bytes': None,
        'error': last_error,
    }

In [7]:
results = []

for url in tqdm(selected_urls):
    result = download_zip_file(url, TEXTOS_RAW, overwrite=OVERWRITE)
    results.append(result)

    # Pausa leve entre arquivos para nao martelar o portal.
    time.sleep(random.uniform(1.0, 2.5))

download_report = pd.DataFrame(results)
download_report_path = REPORTS_DATA / 'stj_textos_download_report.csv'
download_report.to_csv(download_report_path, index=False)

if not download_report.empty:
    print(download_report['status'].value_counts(dropna=False))
    print('Bytes baixados/existentes:', int(download_report['bytes'].fillna(0).sum()))
print('Relatorio:', download_report_path)
download_report.tail()

  0%|          | 0/643 [00:00<?, ?it/s]

status
downloaded        639
already_exists      4
Name: count, dtype: int64
Bytes baixados/existentes: 5407587147
Relatorio: /content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_textos_download_report.csv


,url,source_filename,filename,status,path,bytes,error
638,https://dadosabertos.web.stj.jus.br/dataset/a2...,20231215.zip,textos20231215.zip,downloaded,/content/drive/MyDrive/Mestrado/2026/llms/data...,19707378,None
639,https://dadosabertos.web.stj.jus.br/dataset/a2...,20231218.zip,textos20231218.zip,downloaded,/content/drive/MyDrive/Mestrado/2026/llms/data...,15225891,None
640,https://dadosabertos.web.stj.jus.br/dataset/a2...,20231219.zip,textos20231219.zip,downloaded,/content/drive/MyDrive/Mestrado/2026/llms/data...,13075316,None
641,https://dadosabertos.web.stj.jus.br/dataset/a2...,20231220.zip,textos20231220.zip,downloaded,/content/drive/MyDrive/Mestrado/2026/llms/data...,24963354,None
642,https://dadosabertos.web.stj.jus.br/dataset/a2...,20231221.zip,textos20231221.zip,downloaded,/content/drive/MyDrive/Mestrado/2026/llms/data...,12708552,None


## 5. Conferencia local

In [8]:
local_zip_files = sorted(TEXTOS_RAW.rglob('*.zip'))
local_inventory = pd.DataFrame({
    'arquivo': [p.name for p in local_zip_files],
    'pasta': [p.parent.name for p in local_zip_files],
    'bytes': [p.stat().st_size for p in local_zip_files],
})

print('Arquivos ZIP locais:', len(local_zip_files))
if not local_inventory.empty:
    display(local_inventory.groupby('pasta').agg(arquivos=('arquivo', 'size'), bytes=('bytes', 'sum')).reset_index())
print('Primeiros:', [p.name for p in local_zip_files[:5]])
print('Ultimos:', [p.name for p in local_zip_files[-5:]])

Arquivos ZIP locais: 1192


,pasta,arquivos,bytes
0,2021,243,1993640807
1,2022,165,1389153171
2,2023,231,1989434074
3,2024,239,2295109777
4,2025,242,2573888511
5,2026,71,181726579
6,stj_integras,1,2319982


Primeiros: ['textos20210104.zip', 'textos20210105.zip', 'textos20210106.zip', 'textos20210107.zip', 'textos20210111.zip']
Ultimos: ['textos20260414.zip', 'textos20260415.zip', 'textos20260416.zip', 'textos20260417.zip', 'textos20260319.zip']


## 6. Reexecutar falhas

In [9]:
failed_urls = download_report.loc[download_report['status'].eq('failed'), 'url'].tolist() if 'download_report' in globals() and not download_report.empty else []
print('Falhas para tentar novamente:', len(failed_urls))

retry_results = []
for url in tqdm(failed_urls):
    retry_results.append(download_zip_file(url, TEXTOS_RAW, overwrite=OVERWRITE, attempts=6, base_sleep=10.0))
    time.sleep(random.uniform(2.0, 4.0))

if retry_results:
    retry_report = pd.DataFrame(retry_results)
    retry_report_path = REPORTS_DATA / 'stj_textos_download_retry_report.csv'
    retry_report.to_csv(retry_report_path, index=False)
    print(retry_report['status'].value_counts(dropna=False))
    print('Relatorio retry:', retry_report_path)
    display(retry_report.tail())

Falhas para tentar novamente: 0


0it [00:00, ?it/s]

## 7. Proximos passos

Depois do download, rode `02_validacao_integras_txt.ipynb` para um lote pequeno ou `07_documentos_por_processo_stj.ipynb` para construir a tabela documento-texto em varios lotes.